In [ ]:
!pip install requests pandas

In [ ]:
import requests

url = "https://edamam-food-and-grocery-database.p.rapidapi.com/parser"

querystring = {"ingr":"apple"}

headers = {
    'x-rapidapi-key': "884ba76159mshaf3ad47e9451c88p147152jsn8015414eb7fc",
    'x-rapidapi-host': "edamam-food-and-grocery-database.p.rapidapi.com"
    }

response = requests.request("GET", url, headers=headers, params=querystring)

print(response.text)

{"message":"Endpoint '\/parser' does not exist"}


In [ ]:
import requests

url = "https://edamam-food-and-grocery-database.p.rapidapi.com/api/food-database/v2/nutrients"

payload = { "ingredients": [
		{
			"quantity": 0,
			"measureURI": "",
			"qualifiers": [],
			"foodId": ""
		}
	] }
headers = {
	"x-rapidapi-key": "884ba76159mshaf3ad47e9451c88p147152jsn8015414eb7fc",
	"x-rapidapi-host": "edamam-food-and-grocery-database.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response)
# print(response.json())

<Response [401]>


In [ ]:
import requests
import pandas as pd
import os

url = "https://world.openfoodfacts.org/api/v2/search"

params = {
    "query": "soda",
    "fields": "product_name,brands,code,ingredients_text,ingredients,categories,labels,countries", # We will extract the requested fields from the product data
    "lc": "fr",
    "sort_by": "popularity_key",
    "page_size": 10 # Limit to the first 10 products
}

headers = {
    "User-Agent": "MonAppli/1.0 (sam9741203@gmail.com)" # It's good practice to include a User-Agent
}

try:
    resp = requests.get(url, params=params, headers=headers, timeout=20)
    resp.raise_for_status() # Raise an exception for bad status codes
    data = resp.json()

    products_data = []
    if 'products' in data:
        for product in data['products']:
            # Extract the requested fields, using .get() for safety in case a field is missing
            product_info = {
                'foodId': product.get('code'), # Using 'code' as a potential equivalent for foodId
                'label': product.get('product_name'),
                'category': product.get('categories'),
                'foodContentsLabel': product.get('ingredients_text'), # Using 'ingredients_text' for foodContentsLabel
                'image': product.get('image_url') # OpenFoodFacts uses 'image_url'
            }
            products_data.append(product_info)

    if products_data:
        df_products = pd.DataFrame(products_data)

        # Define the output directory and filename
        output_dir = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output'
        output_filename = 'champagne_products_off.csv'
        output_filepath = os.path.join(output_dir, output_filename)

        # Create the output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Save the DataFrame to a CSV file
        df_products.to_csv(output_filepath, index=False)

        print(f"Successfully extracted {len(products_data)} products and saved to {output_filepath}")
        display(df_products)

    else:
        print("No products found for the query.")

except requests.exceptions.RequestException as e:
    print(f"Error fetching data from OpenFoodFacts API: {e}")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully extracted 10 products and saved to /content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output/champagne_products_off.csv


,foodId,label,category,foodContentsLabel,image
0,6111035000430,Sidi Ali,"Beverages and beverages preparations,Beverages...",OBD1 999 999 1112606 266963207 mb,None
1,6111242100992,Perly,"Dairies,Fermented foods,Fermented milk product...","Lait écrémé, crème, SUcre, ferments laciques M...",None
2,6111035002175,Sidi Ali,"Beverages and beverages preparations,Beverages...",Sodium 26 Calcium 12 Magnésium 9 Potassium 3 B...,None
3,6111035000058,Eau minérale naturelle,"Beverages and beverages preparations,Beverages...",eau minérale naturelle,None
4,6111252421568,اكوافينا,"Boissons et préparations de boissons,Boissons,...",ouverture et avant le : Voir bouteille. après ...,None
5,6111266962187,Lait,"Dairies,Meals,Milks (liquid and powder),Milks,...",Lait frais pasteurisé demi-écrémé à 15g/l de m...,None
6,3274080005003,Eau De Source,"Beverages and beverages preparations,Beverages...",Eau de source,None
7,6111246721261,Fromage blanc nature,"Dairies,Fermented foods,Fermented milk product...","Ferments lactiques, Présure, Sel.",None
8,6111242101180,uht jaouda 1L,"Dairies,Milks,Homogenized milks,UHT Milks,Whol...",Lait entier,None
9,6111242106949,Jben,"en:Dairies, en:Fermented foods, en:Fermented m...","lait frais entier, crème, stabilisants, amidon...",None


In [ ]:
import requests
import pandas as pd
from urllib.parse import urlencode

BASE_URL = "https://world.openfoodfacts.org/cgi/search.pl"

def search_products(
    search_terms="champagne",
    page=1,
    page_size=50,
    countries=None,          # ex: "france"
    categories=None,         # ex: "champagnes" ou "vins" (tags OFF)
    lang="fr",
    # fields=("code","product_name","brands","categories","ingredients_text",
    #         "countries","nutrition_grades","selected_images"),
    fields=("product_name","brands","code","ingredients_text","ingredients","categories","labels","countries", "image"),
    # fields=("foodId", "label", "category", "foodContentsLabel", "image"),
    sort_by=None             # ex: "unique_scans_n" ou "product_name"
):
    params = {
        "search_terms": search_terms,
        "search_simple": 1,       # recherche simple
        "action": "process",
        "json": 1,
        "page": page,
        "page_size": page_size,
        "locale": lang,           # langue de l’interface/résultats
        "fields": ",".join(fields)
    }
    if countries:
        params["countries"] = countries
    if categories:
        params["categories_tags"] = categories
    if sort_by:
        params["sort_by"] = sort_by

    headers = {
        # Bonne pratique: identifiez votre application/contact
        "User-Agent": "MyApp/1.0 (contact: dev@example.com)"
    }

    url = f"{BASE_URL}?{urlencode(params)}"
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    # data contient: products (liste), count, page, page_size, etc.




    products = data.get("products", [])
    print("Total trouvés:", data.get("count"))
    if products:
        df_products = pd.DataFrame(products)
        # Define the output directory and filename
        output_dir = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output'
        output_filename = 'champagne_products_off.csv'
        output_filepath = os.path.join(output_dir, output_filename)

        # Create the output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Save the DataFrame to a CSV file
        df_products.to_csv(output_filepath, index=False)
        print(f"Successfully extracted {len(products_data)} products and saved to {output_filepath}")

        return df_products
    else:
        return None

if __name__ == "__main__":
    # Exemple: produits liés à “champagne” vendus en France, triés par popularité
    df_products = search_products(
        search_terms="champagne",
        countries=None,
        categories="champagnes",            # ou "champagnes"
        page=1,
        page_size=10,
        lang="fr",
        sort_by="unique_scans_n"
    )
    if df_products:
      display(df_products)
    else:
        print("No products found.")

Total trouvés: 645
Successfully extracted 10 products and saved to /content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output/champagne_products_off.csv


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [2]:
import requests
import pandas as pd
import os

urlAPI = "https://world.openfoodfacts.org/cgi/search.pl"
parametres = {
    'search_terms': "Champagne",
    'search_simple': 1,
    'action': 'process',
    'json': 1,
    'page_size': 100
}

response = requests.get(urlAPI, params=parametres)

if(response.status_code == 200):
    donnees = response.json()
    produits = donnees.get("products", [])

    produitsChampagnes = []

    for prod in produits:
        nom = prod.get("product_name", "").lower()
        categories = prod.get("categories_tags", [])

        if("champagne" in nom) and any("en:champagne" in categ for categ in categories):
            produitsChampagnes.append(prod)

    print(len(produitsChampagnes), "produits trouvés (Nom ou Catégorie = \"Champagne\") :\n")

    # Create a list of dictionaries with the desired fields for the first 10 products
    data_to_save = []
    for prod in produitsChampagnes[:10]:
        data_to_save.append({
            "foodId": prod.get("id", "FoodId inconnu"),
            "label": prod.get("product_name", "Label inconnu"),
            "category": prod.get("categories_tags", "Category inconnue"),
            "foodContentsLabel": prod.get("ingredients_text", "foodContentsLabel inconnue"),
            "image": prod.get("image_url", "Image inconnue")
        })

    if data_to_save:
        dataframe = pd.DataFrame(data_to_save)

        # Define the output directory and filename
        output_dir = '/content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output'
        output_filename = 'champagne_products_off.csv'
        output_filepath = os.path.join(output_dir, output_filename)

        # Create the output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Save the DataFrame to a CSV file
        dataframe.to_csv(output_filepath, index=False)

        print(f"\nDataFrame créé avec les {len(data_to_save)} premiers produits et sauvegardé dans {output_filepath} !")
        display(dataframe)

    else:
        print("Aucun produit trouvé correspondant aux critères.")

else:
    print("Erreur HTTP :", response.status_code)

48 produits trouvés (Nom ou Catégorie = "Champagne") :


DataFrame créé avec les 10 premiers produits et sauvegardé dans /content/drive/Othercomputers/Mon ordinateur portable/ingénieur ia/projet 6/dev/output/champagne_products_off.csv !


,foodId,label,category,foodContentsLabel,image
0,3114080034057,Champagne rosé,"[en:beverages, en:alcoholic-beverages, en:wine...",,https://images.openfoodfacts.org/images/produc...
1,3185370729960,Champagne Impérial Brut,"[en:beverages-and-beverages-preparations, en:b...",,https://images.openfoodfacts.org/images/produc...
2,3049610004104,Veuve Clicquot Champagne Ponsardin Brut,"[en:beverages-and-beverages-preparations, en:b...",Champagne,https://images.openfoodfacts.org/images/produc...
3,3113910312013,Champagne Alfred Rothschild et Cie brut,"[en:beverages, en:alcoholic-beverages, en:wine...",Champagne brut (contient _sulfites_),https://images.openfoodfacts.org/images/produc...
4,20712907,Champagne,"[en:beverages, en:alcoholic-beverages, en:wine...",foodContentsLabel inconnue,https://images.openfoodfacts.org/images/produc...
5,3185370283905,Champagne Ruinart,"[en:beverages, en:alcoholic-beverages, en:wine...",champagne,https://images.openfoodfacts.org/images/produc...
6,3416181017169,"Champagne AOP, brut","[en:beverages, en:alcoholic-beverages, en:wine...",Champagne,https://images.openfoodfacts.org/images/produc...
7,3760017736329,Champagne Paul Menand,"[en:beverages, en:alcoholic-beverages, en:wine...",foodContentsLabel inconnue,https://images.openfoodfacts.org/images/produc...
8,3016570001030,75CL Champagne Brut Reserve Taittinger,"[en:beverages, en:alcoholic-beverages, en:wine...",Champagne brut (_sulfites_),https://images.openfoodfacts.org/images/produc...
9,3254560075283,Champagne brut,"[en:beverages, en:alcoholic-beverages, en:wine...",foodContentsLabel inconnue,Image inconnue
